# Phase 2: Sleep Stage Classification

Train an XGBoost classifier on epoch-aligned sensor features to distinguish
REM sleep from other sleep stages. This notebook walks through the full
pipeline: data preparation, feature engineering, model training,
cross-validation, evaluation, and export.

**Success criteria:** REM Precision ≥ 0.70, REM Recall ≥ 0.60, REM F1 ≥ 0.65

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lullaby.loader import generate_mock_session, load_all_sessions
from lullaby.features import align_to_epochs
from lullaby.temporal_features import add_temporal_features, get_temporal_feature_names
from lullaby.pipeline import (
    prepare_sessions, split_by_session, get_leave_one_out_splits,
    generate_mock_dataset, make_binary_labels, STAGE_LABELS, STAGE_TO_INT,
)
from lullaby.model import (
    XGBConfig, train_multiclass, train_binary_rem, cross_validate, tune_hyperparameters,
)
from lullaby.evaluation import (
    compute_metrics, compute_rem_onset_latency, check_success_criteria,
    predict, print_metrics_report,
    plot_confusion_matrix, plot_hypnogram_comparison,
    plot_feature_importance, plot_shap_summary, plot_cv_results,
)
from lullaby.export import save_model, load_model

print("Imports OK")

## 2. Load Data

Use mock data for development. Replace `DATA_DIR` with a path to real
exported JSON sessions when available.

In [ ]:
# Set DATA_DIR to a folder of real .json exports, or None for mock data
DATA_DIR = None
N_MOCK_SESSIONS = 10
MOCK_DURATION_HOURS = 6.0

if DATA_DIR is not None:
    sessions = load_all_sessions(DATA_DIR)
    print(f"Loaded {len(sessions)} real sessions from {DATA_DIR}")
else:
    sessions = [
        generate_mock_session(duration_hours=MOCK_DURATION_HOURS, seed=100 + i)
        for i in range(N_MOCK_SESSIONS)
    ]
    print(f"Generated {len(sessions)} mock sessions ({MOCK_DURATION_HOURS}h each)")

print(f"Session IDs: {[s.session_id[:8] for s in sessions]}")

## 3. Prepare Dataset

Run the full pipeline: quality gating → epoch alignment → temporal features → label encoding.

In [ ]:
dataset = prepare_sessions(sessions, min_coverage_pct=0.0)

print(f"Total epochs: {len(dataset.y)}")
print(f"Features: {len(dataset.feature_names)}")
print(f"Sessions: {len(np.unique(dataset.session_ids))}")
print()

# Class distribution
for label_int in sorted(set(dataset.y)):
    name = dataset.stage_labels[label_int]
    count = (dataset.y == label_int).sum()
    pct = count / len(dataset.y) * 100
    print(f"  {name:12s}: {count:5d} ({pct:.1f}%)")

## 4. Temporal Feature Overview

In [ ]:
temporal_names = get_temporal_feature_names()
print(f"Temporal features added: {len(temporal_names)}")
print(f"Total features: {len(dataset.feature_names)}")
print()
for name in sorted(dataset.feature_names):
    marker = " [temporal]" if name in temporal_names else ""
    print(f"  {name}{marker}")

## 5. Train/Test Split

In [ ]:
split = split_by_session(dataset, test_fraction=0.2, seed=42)

print(f"Train: {len(split.train.y)} epochs from {len(split.train_session_ids)} sessions")
print(f"Test:  {len(split.test.y)} epochs from {len(split.test_session_ids)} sessions")
print(f"Train sessions: {[s[:8] for s in split.train_session_ids]}")
print(f"Test sessions:  {[s[:8] for s in split.test_session_ids]}")

## 6. Train 5-Class Model

In [ ]:
config = XGBConfig(n_estimators=300, max_depth=6, learning_rate=0.1)
model_multi = train_multiclass(split.train, config=config)

# Evaluate on test set
preds_multi = predict(model_multi, split.test)
metrics_multi = compute_metrics(split.test.y, preds_multi)
criteria_multi = check_success_criteria(metrics_multi)

print_metrics_report(metrics_multi, criteria_multi);

## 7. Train Binary REM Model

In [ ]:
model_binary = train_binary_rem(split.train, config=config)

y_test_binary = make_binary_labels(split.test.y)
preds_binary = predict(model_binary, split.test)
metrics_binary = compute_metrics(y_test_binary, preds_binary, binary=True)

print_metrics_report(metrics_binary);

## 8. Cross-Validation (Leave-One-Session-Out)

In [ ]:
cv_result = cross_validate(dataset, config=config)

print("Per-fold results:")
for i, m in enumerate(cv_result.fold_metrics):
    print(f"  Fold {i} (session {m['test_session'][:8]}): "
          f"acc={m['accuracy']:.3f}, REM P={m['rem_precision']:.3f}, "
          f"R={m['rem_recall']:.3f}, F1={m['rem_f1']:.3f}")

print(f"\nMean: acc={cv_result.mean_metrics['accuracy']:.3f}, "
      f"REM F1={cv_result.mean_metrics['rem_f1']:.3f} "
      f"(± {cv_result.std_metrics['rem_f1']:.3f})")

fig = plot_cv_results(cv_result)
fig.savefig("../data/cv_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Confusion Matrix & Hypnogram

In [ ]:
fig_cm = plot_confusion_matrix(metrics_multi)
fig_cm.savefig("../data/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

fig_hyp = plot_hypnogram_comparison(split.test.y, preds_multi)
fig_hyp.savefig("../data/hypnogram_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Feature Importance

In [ ]:
fig_imp = plot_feature_importance(model_multi, max_features=20)
fig_imp.savefig("../data/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. SHAP Analysis

In [ ]:
fig_shap = plot_shap_summary(model_multi, split.test, max_samples=200)
fig_shap.savefig("../data/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. REM Onset Latency

In [ ]:
# Compute on binary predictions for cleaner analysis
rem_label = STAGE_TO_INT["remSleep"]
latency = compute_rem_onset_latency(split.test.y, preds_multi, rem_label=rem_label)

if latency is not None:
    print(f"Mean REM onset latency: {latency:.1f} epochs ({latency * 30:.0f} seconds)")
    if latency <= 3:
        print("PASS: Latency ≤ 3 epochs (90 seconds)")
    else:
        print(f"WARN: Latency > 3 epochs — {latency:.1f} epochs")
else:
    print("No REM bouts found in test set")

## 13. Export Model & Success Gate

In [ ]:
# Final success gate
criteria = check_success_criteria(metrics_multi)
print("=" * 50)
print("PHASE 2 SUCCESS GATE")
print("=" * 50)
print(f"REM Precision >= 0.70: {'PASS' if criteria.rem_precision_met else 'FAIL'} ({metrics_multi.rem_precision:.3f})")
print(f"REM Recall    >= 0.60: {'PASS' if criteria.rem_recall_met else 'FAIL'} ({metrics_multi.rem_recall:.3f})")
print(f"REM F1        >= 0.65: {'PASS' if criteria.rem_f1_met else 'FAIL'} ({metrics_multi.rem_f1:.3f})")
print(f"Overall: {'PASSED' if criteria.passed else 'FAILED'}")

# Save model
output_dir = save_model(model_multi, "../data/model_v1", version="0.1.0", metrics=metrics_multi)
print(f"\nModel saved to: {output_dir}")

# Verify reload
loaded, artifact = load_model(output_dir)
preds_check = loaded.model.predict(split.test.X)
assert np.array_equal(preds_multi, preds_check), "Loaded model predictions differ!"
print("Reload verification: OK")